# Phase C: SFG-BiCross 7 文件格式化

## 目标
将 Phase A + B 的合成数据格式化为 KuaiRand 兼容格式，供 SFG-BiCross 训练。

## 输出 (data/kuairand/)
| # | 文件 | 内容 |
|---|---|---|
| 1 | user_features_pure.csv | 用户纯内容特征（无ID embedding） |
| 2 | video_features_basic_pure.csv | 物品纯内容特征 |
| 3 | log_standard_train_pure.csv | 反馈日志训练集（时间前70%） |
| 4 | log_standard_val_pure.csv | 反馈日志验证集（中间15%） |
| 5 | log_standard_test_pure.csv | 反馈日志测试集（后15%） |
| 6 | log_random_test_pure.csv | 随机曝光日志（Phase 2去偏） |
| 7 | kuairand_video_captions.csv | 物品文本描述 |
| 8 | kuairand_video_categories.csv | 二级分类层级 |

## 字段映射（行为 → KuaiRand）
| 原始 | KuaiRand | 说明 |
|---|---|---|
| view | is_click | 点击/查看 |
| like | is_like | 点赞 |
| fav | is_follow | 收藏→关注 |
| comment | is_comment | 评论 |
| share | is_forward | 分享→转发 |
| — | is_hate | 恒为0（无负反馈） |
| fav or (like+高分) | long_view | 深度互动 |

In [ ]:
import pandas as pd
import numpy as np
import json
from collections import defaultdict
import os, warnings
warnings.filterwarnings('ignore')

DATA_DIR = '../data'
OUT_DIR = '../data/kuairand'
SRC_DIR = 'D:/dev/projects/zhere_olap/data/recommendation'
SEED = 42
np.random.seed(SEED)
rng = np.random.RandomState(SEED)
os.makedirs(OUT_DIR, exist_ok=True)
print('Setup OK')

In [ ]:
# 加载数据
user_latent = pd.read_csv(f'{DATA_DIR}/user_latent.csv', index_col='user_id')
item_latent = pd.read_csv(f'{DATA_DIR}/item_latent.csv', index_col='item_id')
interactions = pd.read_csv(f'{DATA_DIR}/interactions_enriched.csv')

# Fix double prefix
rename_map = {}
for c in user_latent.columns:
    if c.startswith('theme_theme_'):
        rename_map[c] = c.replace('theme_theme_', 'theme_')
user_latent.rename(columns=rename_map, inplace=True)

print(f'Users: {len(user_latent):,}, Items: {len(item_latent):,}, Interactions: {len(interactions):,}')

# 加载原始文本表
asset_df = pd.read_csv(f'{SRC_DIR}/published_asset.csv')
prompt_df = pd.read_csv(f'{SRC_DIR}/published_prompt.csv')
commerce_df = pd.read_csv(f'{SRC_DIR}/published_commerce.csv')

# 构建 item → 文本映射
text_lookup = {}
for _, row in asset_df.iterrows():
    text_lookup[row['asset_id']] = {
        'title': str(row.get('title','') or ''),
        'description': str(row.get('description','') or ''),
        'theme': str(row.get('theme','') or ''),
        'location': str(row.get('location','') or ''),
        'content_type': 'asset',
        'duration': int(row.get('duration',0) or 0),
    }
for _, row in prompt_df.iterrows():
    text_lookup[row['prompt_id']] = {
        'title': str(row.get('title','') or ''),
        'description': str(row.get('description','') or ''),
        'theme': str(row.get('theme','') or ''),
        'location': str(row.get('location','') or ''),
        'content_type': 'prompt',
        'duration': 0,
    }
for _, row in commerce_df.iterrows():
    desc = ' '.join(str(row.get(f,'') or '') for f in ['organization','activity','context','requirements'])
    text_lookup[row['commerce_id']] = {
        'title': str(row.get('activity','') or ''),
        'description': desc,
        'theme': str(row.get('range','') or ''),
        'location': str(row.get('place','') or ''),
        'content_type': 'commerce',
        'duration': 0,
    }
print(f'Text lookup: {len(text_lookup):,} items')

---
## 1. user_features_pure.csv

用户纯内容特征。包含全部 latent factor（theme SVD 50维、skill TF-IDF 32维、ct_pref 3维、quality_sensitivity、budget_level、social_tendency、activity_level）+ 衍生活跃度。**不含 user_id embedding**。

In [ ]:
user_exclude = ['cluster']
user_feature_cols = [c for c in user_latent.columns if c not in user_exclude]
user_features = user_latent[user_feature_cols].reset_index().fillna(0)

# 衍生活跃度
iu = interactions.groupby('user_id').size()
user_features['user_active_degree'] = user_features['user_id'].map(
    lambda u: min(iu.get(u, 0) / 80, 1.0))

user_features.to_csv(f'{OUT_DIR}/user_features_pure.csv', index=False)
print(f'Saved: {len(user_features)} users × {len(user_features.columns)} features')

---
## 2. video_features_basic_pure.csv

物品纯内容特征。包含全部 item latent factor（theme SVD 50维、skill TF-IDF 32维、ct one-hot 3维、quality_score、price_tier、freshness、popularity）+ video_type。**不含 video_id embedding**。

In [ ]:
CT_MAP = {'asset': 1, 'prompt': 2, 'commerce': 3}
item_exclude = ['cluster']
video_feature_cols = [c for c in item_latent.columns if c not in item_exclude]
video_features = item_latent[video_feature_cols].reset_index().fillna(0)
video_features = video_features.rename(columns={'item_id': 'video_id'})

# 添加 video_type
ct_cols = ['ct_published_asset', 'ct_published_prompt', 'ct_published_commerce']
for col in ct_cols:
    if col in video_features.columns:
        ct_type = col.replace('ct_published_', '')
        video_features.loc[video_features[col] == 1, 'video_type'] = CT_MAP.get(ct_type, 0)
video_features['video_type'] = video_features['video_type'].fillna(0).astype(int)

video_features.to_csv(f'{OUT_DIR}/video_features_basic_pure.csv', index=False)
print(f'Saved: {len(video_features)} videos × {len(video_features.columns)} features')
print(f'video_type dist: {video_features["video_type"].value_counts().to_dict()}')

---
## 3-5. 时间切分：Train / Val / Test

按时间排序后 70% / 15% / 15% 切分。各 split 的用户和行为分布保持一致。

In [ ]:
times = pd.to_datetime(interactions['event_time_ms'], unit='ms')
interactions['timestamp'] = times
interactions['date'] = times.dt.strftime('%Y-%m-%d')

t_min, t_max = times.min(), times.max()
t_range = (t_max - t_min).total_seconds()
train_cut = t_min + pd.Timedelta(seconds=t_range * 0.70)
val_cut   = t_min + pd.Timedelta(seconds=t_range * 0.85)

print(f'Time: {t_min} ~ {t_max}')
print(f'Train cutoff: {train_cut}')
print(f'Val cutoff:   {val_cut}')

train_mask = times <= train_cut
val_mask   = (times > train_cut) & (times <= val_cut)
test_mask  = times > val_cut

print(f'Train: {train_mask.sum():,} ({train_mask.mean()*100:.1f}%)')
print(f'Val:   {val_mask.sum():,} ({val_mask.mean()*100:.1f}%)')
print(f'Test:  {test_mask.sum():,} ({test_mask.mean()*100:.1f}%)')

In [ ]:
def build_log(df_split):
    log = pd.DataFrame()
    log['user_id']   = df_split['user_id']
    log['video_id']  = df_split['item_id']
    log['date']      = df_split['date']
    log['time_ms']   = df_split['event_time_ms']
    log['is_click']  = df_split['view'].astype(int)
    log['is_like']   = df_split['like'].astype(int)
    log['is_follow'] = df_split['fav'].astype(int)
    log['is_comment']= df_split['comment'].astype(int)
    log['is_forward']= df_split['share'].astype(int)
    log['is_hate']   = 0
    # Plan C: long_view is an independent behavior from Phase B
    log['long_view'] = df_split['long_view'].astype(int)
    log['play_time_ms'] = (df_split['match_score'] * 120000).astype(int)
    log['duration_ms']  = df_split['item_id'].map(
        lambda iid: text_lookup.get(iid, {}).get('duration', 60) * 1000).fillna(60000).astype(int)
    log['profile_stay_time'] = 0
    log['comment_stay_time'] = df_split['comment'].apply(lambda x: rng.randint(5000,30000) if x==1 else 0)
    return log

log_train = build_log(interactions[train_mask])
log_val   = build_log(interactions[val_mask])
log_test  = build_log(interactions[test_mask])

log_train.to_csv(f'{OUT_DIR}/log_standard_train_pure.csv', index=False)
log_val.to_csv(  f'{OUT_DIR}/log_standard_val_pure.csv',   index=False)
log_test.to_csv( f'{OUT_DIR}/log_standard_test_pure.csv',  index=False)

print(f'Saved: train={len(log_train):,}, val={len(log_val):,}, test={len(log_test):,}')
print(f'long_view (independent): train={log_train["long_view"].mean()*100:.1f}% '
      f'val={log_val["long_view"].mean()*100:.1f}% test={log_test["long_view"].mean()*100:.1f}%')

---
## 6. log_random_test_pure.csv

随机曝光日志——每用户随机分配 15-25 个未匹配 item。用于 SFG-BiCross Phase 2 去偏训练。标记 `is_random=1`。

In [ ]:
all_item_ids = item_latent.index.values
all_user_ids = user_latent.index.values

random_records = []
test_start = train_cut + pd.Timedelta(seconds=1)

for uid in all_user_ids:
    n_random = rng.randint(15, 26)
    random_items = rng.choice(all_item_ids, size=n_random, replace=False)
    test_end = t_max
    test_secs = (test_end - test_start).total_seconds()
    random_offsets = rng.randint(0, int(test_secs), size=n_random)
    random_times = [test_start + pd.Timedelta(seconds=int(s)) for s in random_offsets]

    for iid, ts in zip(random_items, random_times):
        random_records.append({
            'user_id': uid, 'video_id': iid,
            'date': ts.strftime('%Y-%m-%d'),
            'time_ms': int(ts.timestamp() * 1000),
            'is_click': 0, 'is_like': 0, 'is_follow': 0,
            'is_comment': 0, 'is_forward': 0, 'is_hate': 0,
            'long_view': 0, 'play_time_ms': 0,
            'duration_ms': text_lookup.get(iid, {}).get('duration', 60) * 1000,
            'profile_stay_time': 0, 'comment_stay_time': 0,
            'is_random': 1,
        })

log_random = pd.DataFrame(random_records)
log_random.to_csv(f'{OUT_DIR}/log_random_test_pure.csv', index=False)
print(f'Saved: {len(log_random):,} random exposures ({len(log_random)/len(all_user_ids):.1f}/user)')

---
## 7. kuairand_video_captions.csv

物品文本：caption_text = title + " " + description（截断512字符），cover_text = theme + " " + location。

In [ ]:
captions = []
for iid in item_latent.index:
    info = text_lookup.get(iid, {})
    caption_text = f"{info.get('title','')} {info.get('description','')}".strip()[:512]
    cover_text = f"{info.get('theme','')} {info.get('location','')}".strip()[:256]
    captions.append({'video_id': iid, 'caption_text': caption_text, 'cover_text': cover_text})

captions_df = pd.DataFrame(captions)
captions_df.to_csv(f'{OUT_DIR}/kuairand_video_captions.csv', index=False)
print(f'Saved: {len(captions_df):,} captions')
print(f'Non-empty: {(captions_df["caption_text"].str.len()>0).sum():,}')

---
## 8. kuairand_video_categories.csv

二级分类：Level 1 = content_type (Asset/Prompt/Commerce)，Level 2 = theme/location。

In [ ]:
categories = []
for iid in item_latent.index:
    info = text_lookup.get(iid, {})
    level1 = info.get('content_type', 'unknown').capitalize()
    level2 = info.get('theme', '') or info.get('location', '') or 'Other'
    categories.append({'video_id': iid, 'category_level_1': level1, 'category_level_2': level2})

categories_df = pd.DataFrame(categories)
categories_df.to_csv(f'{OUT_DIR}/kuairand_video_categories.csv', index=False)

print(f'Saved: {len(categories_df):,} categories')
print(f'Level 1: {categories_df["category_level_1"].value_counts().to_dict()}')
print(f'Level 2 unique: {categories_df["category_level_2"].nunique()}')

---
## 验证

In [ ]:
print('='*60)
print('Phase C Output Summary')
print('='*60)
files = [
    ('user_features_pure.csv', user_features),
    ('video_features_basic_pure.csv', video_features),
    ('log_standard_train_pure.csv', log_train),
    ('log_standard_val_pure.csv', log_val),
    ('log_standard_test_pure.csv', log_test),
    ('log_random_test_pure.csv', log_random),
    ('kuairand_video_captions.csv', captions_df),
    ('kuairand_video_categories.csv', categories_df),
]
for name, df in files:
    path = f'{OUT_DIR}/{name}'
    size_mb = os.path.getsize(path) / 1024**2
    print(f'  {name:<38} {len(df):>8,} rows × {len(df.columns):>3} cols  [{size_mb:.1f} MB]')

print(f'\nTrain/Val/Test: {len(log_train):,} / {len(log_val):,} / {len(log_test):,}')

# 行为分布一致性
for label, log in [('train', log_train), ('val', log_val), ('test', log_test)]:
    print(f'{label}: click={log["is_click"].mean()*100:.1f}% like={log["is_like"].mean()*100:.1f}% '
          f'follow={log["is_follow"].mean()*100:.1f}% comment={log["is_comment"].mean()*100:.1f}% '
          f'forward={log["is_forward"].mean()*100:.1f}% long_view={log["long_view"].mean()*100:.1f}%')

print(f'\nRandom log: {len(log_random):,} exposures, is_random={log_random["is_random"].sum():,}')
print(f'Time: {log_train["date"].min()} → {log_train["date"].max()} | '
      f'{log_val["date"].min()} → {log_val["date"].max()} | '
      f'{log_test["date"].min()} → {log_test["date"].max()}')
print('\nPhase C Complete ✓')